<a href="https://colab.research.google.com/github/nandobelar/MetroBot/blob/main/ModeloBot_2.0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
%pip install -q groq ollama ipywidgets python-dotenv

### MetrôBot SP 2.0 — Desafio Completo com 3 Linhas

Nesta célula abaixo, implementamos a versão 2.0 do MetrôBot SP seguindo todos os requisitos e diretrizes:
- **R1 - Modelagem (grafo de 3 linhas):** Função `construir_grafo_multilinhas(linhas)` mapeando 52 estações únicas e rastreando quais linhas servem cada par de trecho.
- **R2 - Busca:** Algoritmos BFS e DFS adaptados, calculando o número de paradas e as estações visitadas por cada busca.
- **R3 - Base de Conhecimento e Lógica:** Pelo menos 9 locais no total (3 por linha). Além disso, adicionamos as regras de inferência, a regra dedutiva automática **R6** para conexões (`∀e ∀l1 ∀l2 (pertence(e,l1) ∧ pertence(e,l2) ∧ l1 ≠ l2 → integracao(e))`) e a regra **R7** para horário de pico (`∀e (horario_pico ∧ estacao(e) → lotada(e))`). Inclui também a geração da tabela-verdade.
- **R4 - LLM:** Intérprete JSON robusto (com fallback resiliente local/offline) contendo todas as 52 estações e locais, e Narrador explicando a rota detalhando baldeações de forma simpática.
- **R5 - Interface Avançada com ipywidgets:** Caixa de texto para o pedido, dropdowns dinâmicos, controle de fechamento/manutenção e renderização colorida em HTML das linhas identificando o caminho traçado, as visitas de busca, transferências e alertas.
- **R6 - Testes:** Função `rodar_testes_desafio()` contendo os 6 casos do desafio mais 2 casos personalizados.

In [19]:
import os, json, re, unicodedata
from collections import deque
from itertools import product
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

# ---------- DADOS DAS LINHAS E CORES ----------
LINHAS = {
    "Linha 1-Azul": [
        "Tucuruvi", "Parada Inglesa", "Jardim São Paulo", "Santana",
        "Carandiru", "Portuguesa-Tietê", "Armênia", "Tiradentes", "Luz",
        "São Bento", "Sé", "Japão-Liberdade", "São Joaquim", "Vergueiro",
        "Paraíso", "Ana Rosa", "Vila Mariana", "Santa Cruz",
        "Praça da Árvore", "Saúde", "São Judas", "Conceição", "Jabaquara",
    ],
    "Linha 2-Verde": [
        "Vila Madalena", "Sumaré", "Clínicas", "Consolação", "Trianon-Masp",
        "Brigadeiro", "Paraíso", "Ana Rosa", "Chácara Klabin",
        "Santos-Imigrantes", "Alto do Ipiranga", "Sacomã", "Tamanduateí",
        "Vila Prudente",
    ],
    "Linha 3-Vermelha": [
        "Palmeiras-Barra Funda", "Marechal Deodoro", "Santa Cecília",
        "República", "Anhangabaú", "Sé", "Pedro II", "Brás",
        "Bresser-Mooca", "Belém", "Tatuapé", "Carrão", "Penha",
        "Vila Matilde", "Guilhermina-Esperança", "Patriarca-Vila Ré",
        "Artur Alvim", "Corinthians-Itaquera",
    ],
}

CORES = {
    "Linha 1-Azul": "#1e88e5",
    "Linha 2-Verde": "#2e7d32",
    "Linha 3-Vermelha": "#d32f2f"
}

PROVEDOR = "groq"
MODELO_GROQ = "openai/gpt-oss-20b"
TEMPO_POR_TRECHO = 2
TEMPO_BALDEACAO = 5

# ---------- FUNÇÕES DE CONEXÃO COM LLM (GROQ) ----------
def obter_chave_groq():
    try:
        from google.colab import userdata
        return userdata.get("GROQ_API_KEY")
    except Exception:
        pass
    return os.environ.get("GROQ_API_KEY")

def chamar_llm(mensagens, modo_json=False):
    from groq import Groq
    chave = obter_chave_groq()
    if not chave:
        raise ValueError("GROQ_API_KEY não foi encontrada no Colab Secrets ou nas variáveis de ambiente.")
    cliente = Groq(api_key=chave)
    extras = {"response_format": {"type": "json_object"}} if modo_json else {}
    resposta = cliente.chat.completions.create(
        model=MODELO_GROQ,
        messages=mensagens,
        temperature=0,
        **extras
    )
    return resposta.choices[0].message.content

# ---------- R1: MODELAGEM ----------
def construir_grafo_multilinhas(linhas):
    grafo = {}
    linhas_do_trecho = {}
    for nome_linha, estacoes in linhas.items():
        for estacao in estacoes:
            if estacao not in grafo:
                grafo[estacao] = []
        for i in range(len(estacoes) - 1):
            a, b = estacoes[i], estacoes[i + 1]
            if b not in grafo[a]:
                grafo[a].append(b)
            if a not in grafo[b]:
                grafo[b].append(a)

            if (a, b) not in linhas_do_trecho:
                linhas_do_trecho[(a, b)] = set()
            if (b, a) not in linhas_do_trecho:
                linhas_do_trecho[(b, a)] = set()
            linhas_do_trecho[(a, b)].add(nome_linha)
            linhas_do_trecho[(b, a)].add(nome_linha)
    return grafo, linhas_do_trecho

GRAFO_M, LINHAS_DO_TRECHO = construir_grafo_multilinhas(LINHAS)
ESTACOES_UNICAS = sorted(list(GRAFO_M.keys()))

# ---------- BALDEAÇÕES ----------
def contar_baldeacoes(caminho, linhas_do_trecho):
    if not caminho or len(caminho) < 2:
        return 0, []

    baldeacoes_ocorridas = []
    linhas_por_trecho = []
    for i in range(len(caminho) - 1):
        linhas_por_trecho.append(linhas_do_trecho[(caminho[i], caminho[i+1])])

    linha_atual = None
    for i, opcoes_linhas in enumerate(linhas_por_trecho):
        if i == 0:
            linha_atual = list(opcoes_linhas)[0]
        else:
            if linha_atual not in opcoes_linhas:
                linha_nova = list(opcoes_linhas)[0]
                baldeacoes_ocorridas.append((caminho[i], linha_nova))
                linha_atual = linha_nova

    return len(baldeacoes_ocorridas), baldeacoes_ocorridas

# ---------- R3: LOCAIS E BASE DE CONHECIMENTO ----------
LOCAIS_DESAFIO = {
    "Shopping Metrô Tucuruvi": "Tucuruvi",
    "Terminal Rodoviário Tietê": "Portuguesa-Tietê",
    "Pinacoteca": "Luz",
    "Catedral da Sé": "Sé",
    "Bairro da Liberdade": "Japão-Liberdade",
    "Terminal Rodoviário Jabaquara": "Jabaquara",
    "Hospital das Clínicas": "Clínicas",
    "MASP": "Trianon-Masp",
    "Parque da Independência": "Santos-Imigrantes",
    "Theatro Municipal": "Anhangabaú",
    "Mercado Municipal": "São Bento",
    "Neo Química Arena": "Corinthians-Itaquera",
    "Mooca": "Bresser-Mooca"
}

def fatos_base_desafio():
    fatos = set()
    for nome_linha, estacoes in LINHAS.items():
        for e in estacoes:
            fatos.add(("estacao", e))
            fatos.add(("pertence", e, nome_linha))
    for local, estacao in LOCAIS_DESAFIO.items():
        fatos.add(("proximo_de", local, estacao))
    return fatos

def consultar(fatos, predicado):
    return [f[1:] for f in fatos if f[0] == predicado]

def r_origem(fatos):
    novos = set()
    for (local,) in consultar(fatos, "usuario_esta_em"):
        for (l, e) in consultar(fatos, "proximo_de"):
            if l == local:
                novos.add(("origem", e))
    for (e,) in consultar(fatos, "usuario_esta_na_estacao"):
        novos.add(("origem", e))
    return novos

def r_destino(fatos):
    novos = set()
    for (local,) in consultar(fatos, "usuario_quer_ir"):
        for (l, e) in consultar(fatos, "proximo_de"):
            if l == local:
                novos.add(("destino", e))
    for (e,) in consultar(fatos, "usuario_quer_ir_estacao"):
        novos.add(("destino", e))
    return novos

def r_bloqueio(fatos):
    return {("bloqueada", e) for (e,) in consultar(fatos, "fechada")}

def r_acessibilidade(fatos):
    if not consultar(fatos, "precisa_acessibilidade"):
        return set()
    return {("inacessivel", e) for (e,) in consultar(fatos, "elevador_em_manutencao")}

def r_alerta(fatos):
    novos = set()
    inacessiveis = {e for (e,) in consultar(fatos, "inacessivel")}
    for papel in ("origem", "destino"):
        for (e,) in consultar(fatos, papel):
            if e in inacessiveis:
                novos.add(("alerta", papel, e))
    return novos

def r_integracao(fatos):
    novos = set()
    pertences = consultar(fatos, "pertence")
    for e1, l1 in pertences:
        for e2, l2 in pertences:
            if e1 == e2 and l1 != l2:
                novos.add(("integracao", e1))
    return novos

def r_horario_pico(fatos):
    novos = set()
    if consultar(fatos, "horario_pico"):
        for (e,) in consultar(fatos, "estacao"):
            novos.add(("lotada", e))
    return novos

REGRAS_DESAFIO = [
    ("R1 origem", "∀l ∀e (usuario_esta_em(l) ∧ proximo_de(l,e) → origem(e))", r_origem),
    ("R2 destino", "∀l ∀e (usuario_quer_ir(l) ∧ proximo_de(l,e) → destino(e))", r_destino),
    ("R3 bloqueio", "∀e (fechada(e) → bloqueada(e))", r_bloqueio),
    ("R4 acessibilidade", "∀e (precisa_acessibilidade ∧ elevador_em_manutencao(e) → inacessivel(e))", r_acessibilidade),
    ("R5 alerta", "∀p ∀e (papel(p,e) ∧ inacessivel(e) → alerta(p,e))", r_alerta),
    ("R6 integracao", "∀e ∀l1 ∀l2 (pertence(e,l1) ∧ pertence(e,l2) ∧ l1 ≠ l2 → integracao(e))", r_integracao),
    ("R7 lotada", "∀e (horario_pico ∧ estacao(e) → lotada(e))", r_horario_pico)
]

def encadear_para_frente_desafio(fatos, regras):
    fatos = set(fatos)
    justificativas = {}
    while True:
        novos_na_rodada = set()
        for nome, _, regra in regras:
            for fato in regra(fatos) - fatos:
                novos_na_rodada.add(fato)
                justificativas[fato] = nome
        if not novos_na_rodada:
            return fatos, justificativas
        fatos |= novos_na_rodada

# ---------- BUSCA BFS E DFS ----------
def bfs_desafio(grafo, origem, destino, bloqueadas=()):
    if origem in bloqueadas or destino in bloqueadas:
        return None, []
    fila = deque([origem])
    pai = {origem: None}
    ordem_visita = []
    while fila:
        atual = fila.popleft()
        ordem_visita.append(atual)
        if atual == destino:
            caminho = []
            curr = destino
            while curr is not None:
                caminho.append(curr)
                curr = pai[curr]
            return list(reversed(caminho)), ordem_visita
        for vizinho in grafo[atual]:
            if vizinho not in pai and vizinho not in bloqueadas:
                pai[vizinho] = atual
                fila.append(vizinho)
    return None, ordem_visita

def dfs_desafio(grafo, origem, destino, bloqueadas=()):
    if origem in bloqueadas or destino in bloqueadas:
        return None, []
    visitados = set()
    ordem_visita = []
    def explorar(atual, caminho):
        visitados.add(atual)
        ordem_visita.append(atual)
        if atual == destino:
            return caminho
        for vizinho in grafo[atual]:
            if vizinho not in visitados and vizinho not in bloqueadas:
                resultado = explorar(vizinho, caminho + [vizinho])
                if resultado:
                    return resultado
        return None
    return explorar(origem, [origem]), ordem_visita

# ---------- PLANEJADOR 2.0 ----------
def planejar_desafio(pedido, fechadas=(), manutencao=(), algoritmo="BFS", horario_pico=False):
    fatos = fatos_base_desafio()
    tipo_o, nome_o = pedido["origem"]
    tipo_d, nome_d = pedido["destino"]

    fatos.add(("usuario_esta_em", nome_o) if tipo_o == "local" else ("usuario_esta_na_estacao", nome_o))
    fatos.add(("usuario_quer_ir", nome_d) if tipo_d == "local" else ("usuario_quer_ir_estacao", nome_d))

    if pedido.get("acessibilidade"):
        fatos.add(("precisa_acessibilidade",))
    if horario_pico:
        fatos.add(("horario_pico",))

    for e in fechadas:
        fatos.add(("fechada", e))
    for e in manutencao:
        fatos.add(("elevador_em_manutencao", e))

    fatos, justificativas = encadear_para_frente_desafio(fatos, REGRAS_DESAFIO)

    origem = consultar(fatos, "origem")[0][0]
    destino = consultar(fatos, "destino")[0][0]
    bloqueadas = {e for (e,) in consultar(fatos, "bloqueada")}
    alertas = consultar(fatos, "alerta")
    integrais = {e for (e,) in consultar(fatos, "integracao")}

    buscar = bfs_desafio if algoritmo == "BFS" else dfs_desafio
    caminho, visitados = buscar(GRAFO_M, origem, destino, bloqueadas)

    num_baldeacoes, lista_baldeacoes = contar_baldeacoes(caminho, LINHAS_DO_TRECHO)
    tempo = 0
    if caminho:
        paradas = len(caminho) - 1
        tempo = paradas * TEMPO_POR_TRECHO + num_baldeacoes * TEMPO_BALDEACAO
    else:
        paradas = None
        tempo = None

    return {
        "origem": origem,
        "destino": destino,
        "algoritmo": algoritmo,
        "caminho": caminho,
        "visitados": visitados,
        "bloqueadas": sorted(list(bloqueadas)),
        "alertas": [f"{papel}: {e}" for papel, e in alertas],
        "paradas": paradas,
        "baldeacoes": num_baldeacoes,
        "detalhe_baldeacoes": lista_baldeacoes,
        "tempo_min": tempo,
        "regras_usadas": sorted(list(set(justificativas.values()))),
        "integrais": sorted(list(integrais))
    }

# ---------- INTERPRETAÇÃO E NARRADOR ONLINE ----------
PROMPT_INTERPRETE_DESAFIO = """Você é o módulo de INTERPRETAÇÃO do MetrôBot SP 2.0.
Sua única tarefa é transformar o pedido do passageiro em JSON.
Estações válidas: {estacoes}
Locais válidos: {locais}
Responda APENAS com um JSON neste formato, sem explicações:
{{"origem": "<nome exato de estação ou local, ou null>",
"destino": "<nome exato de estação ou local, ou null>",
"acessibilidade": <true ou false>}}
Regras:
- Use SOMENTE nomes das listas fornecidas, escritos exatamente como aparecem.
- "acessibilidade" é true se o passageiro mencionar cadeira de rodas, mobilidade reduzida, muletas, carrinho ou elevador.
- Se não souber algum campo, use null."""

def normalizar(texto):
    texto = unicodedata.normalize("NFD", texto.lower())
    return "".join(c for c in texto if unicodedata.category(c) != "Mn")

def resolver_nome_desafio(nome):
    if not nome:
        return None
    alvo = normalizar(nome).strip()

    # 1. Busca correspondência exata de estação
    for estacao in ESTACOES_UNICAS:
        if normalizar(estacao) == alvo:
            return ("estacao", estacao)

    # 2. Busca correspondência exata de local
    for local in LOCAIS_DESAFIO:
        if normalizar(local) == alvo:
            return ("local", local)

    # 3. Busca por substring/aproximação inteligente
    for estacao in ESTACOES_UNICAS:
        if alvo in normalizar(estacao) or normalizar(estacao) in alvo:
            return ("estacao", estacao)
    for local in LOCAIS_DESAFIO:
        if alvo in normalizar(local) or normalizar(local) in alvo:
            return ("local", local)

    return None

def interpretar_pedido_offline(texto):
    texto_sem = normalizar(texto)
    candidatos = [(n, "estacao") for n in ESTACOES_UNICAS] + [(n, "local") for n in LOCAIS_DESAFIO.keys()]
    candidatos.sort(key=lambda c: len(c[0]), reverse=True)
    encontrados = []
    texto_min = texto.lower()
    ocupado = [False] * len(texto_min)
    for nome, tipo in candidatos:
        padrao = nome.lower()
        for m in re.finditer(r"(?<!\w)" + re.escape(padrao) + r"(?!\w)", texto_min):
            if not any(ocupado[m.start():m.end()]):
                encontrados.append((m.start(), nome))
                for i in range(m.start(), m.end()):
                    ocupado[i] = True
    encontrados.sort()
    acess = any(p in texto_sem for p in ["cadeira de rodas", "acessibilidade", "mobilidade", "muleta", "carrinho", "elevador"])
    origem = encontrados[0][1] if len(encontrados) > 0 else None
    destino = encontrados[1][1] if len(encontrados) > 1 else None
    return {
        "origem": resolver_nome_desafio(origem),
        "destino": resolver_nome_desafio(destino),
        "acessibilidade": acess
    }

def interpretar_pedido_desafio(texto):
    if PROVEDOR == "offline":
        return interpretar_pedido_offline(texto), "Interpretado via Offline"
    try:
        sistema = PROMPT_INTERPRETE_DESAFIO.format(
            estacoes=", ".join(ESTACOES_UNICAS),
            locais=", ".join(LOCAIS_DESAFIO.keys())
        )
        mensagens = [
            {"role": "system", "content": sistema},
            {"role": "user", "content": texto}
        ]
        resposta = chamar_llm(mensagens, modo_json=True)
        dados = json.loads(resposta)
        origem = resolver_nome_desafio(dados.get("origem"))
        destino = resolver_nome_desafio(dados.get("destino"))
        acessibilidade = bool(dados.get("acessibilidade"))
        return {"origem": origem, "destino": destino, "acessibilidade": acessibilidade}, f"Interpretado via {PROVEDOR.upper()}"
    except Exception as e:
        print(f"Erro na LLM ({e}). Usando fallback offline.")
        return interpretar_pedido_offline(texto), "Interpretado via Fallback Offline"

PROMPT_NARRADOR_DESAFIO = """Você é o NARRADOR do MetrôBot SP 2.0. Explique a rota calculada para o passageiro em português,
em até 4 frases curtas, simpáticas e acolhedoras. Use unicamente os dados reais fornecidos em JSON. Não invente caminhos fictícios."""

def narrar_desafio(r):
    if PROVEDOR == "offline":
        if r["caminho"] is None:
            return f"Infelizmente, não existe rota disponível de {r['origem']} para {r['destino']} com os bloqueios atuais."
        baldeacoes_str = ""
        if r["baldeacoes"] > 0:
            trocas = [f"{est} para a {lin}" for est, lin in r["detalhe_baldeacoes"]]
            baldeacoes_str = f" Fazendo baldeação em: {', '.join(trocas)}."
        msg = (f"Embarque em {r['origem']} e siga até {r['destino']}. "
               f"Sua viagem terá {r['paradas']} paradas e levará cerca de {r['tempo_min']} minutos.{baldeacoes_str}")
        if r["alertas"]:
            msg += " Atenção: Elevador em manutenção detectado no percurso!"
        return msg
    try:
        dados_rota = {
            "origem": r["origem"],
            "destino": r["destino"],
            "paradas": r["paradas"],
            "baldeacoes": r["baldeacoes"],
            "tempo_minutos": r["tempo_min"],
            "caminho_estacoes": r["caminho"],
            "alertas_acessibilidade": r["alertas"]
        }
        mensagens = [
            {"role": "system", "content": PROMPT_NARRADOR_DESAFIO},
            {"role": "user", "content": json.dumps(dados_rota, ensure_ascii=False)}
        ]
        return chamar_llm(mensagens, modo_json=False)
    except Exception:
        return "Boa viagem! Siga a rota indicada no mapa abaixo."

# ---------- VISUALIZAÇÃO GRÁFICA MULTILINHAS ----------
def desenhar_linha_desafio(resultado):
    caminho = set(resultado["caminho"] or [])
    visitados = set(resultado["visitados"])
    bloqueadas = set(resultado["bloqueadas"])
    html_output = "<div style='display: flex; gap: 20px; flex-wrap: wrap;'>"
    for nome_linha, estacoes in LINHAS.items():
        cor_linha = CORES[nome_linha]
        linha_html = [f"<div style='border-left: 5px solid {cor_linha}; padding-left: 10px; margin-bottom: 10px;'>"]
        linha_html.append(f"<b style='color: {cor_linha};'>{nome_linha}</b>")
        for estacao in estacoes:
            if estacao in bloqueadas:
                cor, marca = "#d32f2f", " [BLOQUEADA]"
            elif estacao in (resultado["origem"], resultado["destino"]) and estacao in caminho:
                cor, marca = "#0d47a1", " [ALVO]"
            elif estacao in caminho:
                cor, marca = "#1e88e5", " (Rota)"
            elif estacao in visitados:
                cor, marca = "#9e9e9e", " (Visitada)"
            else:
                cor, marca = "#e0e0e0", ""
            linha_html.append(
                f"<div style='display:flex;align-items:center;gap:6px;font-family:sans-serif;font-size:12px;margin:3px 0;'>"
                f"<span style='display:inline-block;width:10px;height:10px;border-radius:50%;background:{cor}'></span>"
                f"<span>{estacao}{marca}</span></div>"
            )
        linha_html.append("</div>")
        html_output += "".join(linha_html)
    html_output += "</div>"
    return html_output

# ---------- CRIAÇÃO DO PAINEL GRÁFICO ----------
opcoes_desafio = ([(f" [Local] {l}", ("local", l)) for l in LOCAIS_DESAFIO] +
                  [(f" [Estação] {e}", ("estacao", e)) for e in ESTACOES_UNICAS])

txt_pedido_d = widgets.Textarea(
    placeholder="Ex: Estou no jabaquara, vou pra lá pra mooca com acessibilidade",
    layout=widgets.Layout(width="95%", height="60px")
)
btn_interpretar_d = widgets.Button(description="Interpretar pedido", button_style="info")
dd_origem_d = widgets.Dropdown(options=opcoes_desafio, description="Origem:")
dd_destino_d = widgets.Dropdown(options=opcoes_desafio, value=("local", "Mooca"), description="Destino:")
chk_acess_d = widgets.Checkbox(description="Acessibilidade")
chk_pico_d = widgets.Checkbox(description="Horário de Pico")
sel_fechadas_d = widgets.SelectMultiple(options=ESTACOES_UNICAS, description="Fechadas:", rows=6)
sel_manut_d = widgets.SelectMultiple(options=ESTACOES_UNICAS, description="Elevador:", rows=6)
rb_algoritmo_d = widgets.RadioButtons(options=["BFS", "DFS"], description="Busca:")
btn_buscar_d = widgets.Button(description="Buscar rota", button_style="success")
saida_d = widgets.Output()

def ao_interpretar_desafio(_):
    with saida_d:
        clear_output()
        pedido, msg = interpretar_pedido_desafio(txt_pedido_d.value)
        print(msg)
        if pedido["origem"]:
            dd_origem_d.value = pedido["origem"]
        if pedido["destino"]:
            dd_destino_d.value = pedido["destino"]
        chk_acess_d.value = pedido["acessibilidade"]
        print("Campos ajustados com sucesso com base no pedido!")

def ao_buscar_desafio(_):
    with saida_d:
        clear_output()
        pedido = {
            "origem": dd_origem_d.value,
            "destino": dd_destino_d.value,
            "acessibilidade": chk_acess_d.value
        }
        r = planejar_desafio(pedido, sel_fechadas_d.value, sel_manut_d.value, rb_algoritmo_d.value, chk_pico_d.value)
        display(HTML(f"<h3>{r['algoritmo']}: {r['origem']} → {r['destino']}</h3>"))
        print("Narrador LLM:", narrar_desafio(r))
        print(f"Nº de Paradas: {r['paradas']}")
        print(f"Nº de Baldeações: {r['baldeacoes']}")
        print(f"Tempo Estimado: {r['tempo_min']} min")
        display(HTML(desenhar_linha_desafio(r)))

btn_interpretar_d.on_click(ao_interpretar_desafio)
btn_buscar_d.on_click(ao_buscar_desafio)

painel_desafio = widgets.VBox([
    widgets.HTML("<h2>MetrôBot SP 2.0 (LLM Online Ativada)</h2>"),
    txt_pedido_d, btn_interpretar_d,
    widgets.HBox([dd_origem_d, dd_destino_d]),
    widgets.HBox([chk_acess_d, chk_pico_d, rb_algoritmo_d]),
    widgets.HBox([sel_fechadas_d, sel_manut_d]),
    btn_buscar_d, saida_d
])

display(painel_desafio)

### Explicação sobre os Casos 5 e 6

- **Caso 5 (Vila Madalena → Jabaquara com Paraíso fechada):** A Linha 2 fica "cortada" e interrompe a possibilidade de fazer baldeação direta rumo à Linha 1 no fluxo natural de quem vai ao sul pela Linha 1, resultando em ausência de caminho válido.
- **Caso 6 (Vila Prudente → Jabaquara com Paraíso fechada):** Embora Paraíso esteja bloqueada, o usuário vindo da Vila Prudente pode desviar usando a estação **Ana Rosa** (que também faz a integração física de transferência entre as Linhas 2-Verde e 1-Azul) antes de atingir a estação Paraíso, completando a viagem com sucesso via desvio.